# Performance-Benchmark: Dijkstra-Router vs. A*-Router

Dieses Notebook führt einen direkten Performance- und Korrektheitsvergleich zwischen dem Standard-**Dijkstra-Router** (ohne Heuristik) und dem optimierten **A*-Router** (mit rückwärtsgerichteter Hub-Distanz-Heuristik) durch.

Beide Algorithmen werden auf demselben zeitexpandierten Graphen aufgerufen, um zu belegen:
1. **Mathematische Äquivalenz:** Dass $A^*$ exakt dieselben optimalen Pfade und denselben Zielfunktionswert findet (Optimality Gap = 0,00%).
2. **Rechenzeit-Ersparnis:** Welcher Beschleunigungsfaktor (Speedup) durch das zielgerichtete $A^*$-Suchverfahren auf dem großen realen Netzwerk erzielt wird.

## 1. Setup und Imports

In [ ]:
import sys
import time
import random
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType
from freight_routing.model import TimeExpandedNetwork
from heuristics.dijkstra_router import DijkstraRouter, AStarRouter

sns.set_theme(style="whitegrid", palette="muted")

## 2. Laden der Netzwerkdaten

Wir laden das große Netzwerk (`large_network.json`) mit 870 Hubs und über 36.000 statischen Verbindungen.

In [ ]:
network_path = PROJECT_ROOT / "dataset/large_network.json"
network_data = NetworkDataLoader.from_json(network_path)
print(f"Netzwerk geladen: {len(network_data.hubs)} Hubs, {len(network_data.arc_templates)} Bogen-Templates.")

## 3. Generierung der Test-Sendungen

Wir generieren eine Stichprobe von **100 Sendungen** mit zufälligen Start- und Zielhubs sowie variierenden Gewichten und Deadlines.

In [ ]:
random.seed(42)
N_SHIPMENTS = 100
PLANNING_DAYS = 30
DEADLINE = PLANNING_DAYS * 24 * 60
hubs_list = list(network_data.hubs.keys())

shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(hubs_list)
    dest = random.choice(hubs_list)
    while dest == start:
        dest = random.choice(hubs_list)
        
    shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=0,
            deadline=DEADLINE,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(1, 20)),
            objective_weights=ObjectiveWeights(cost=0.4, time=0.2, emissions=0.4)
        )
    )

network = TimeExpandedNetwork.build(network_data, planning_days=PLANNING_DAYS, shipments=shipments)
print(f"{N_SHIPMENTS} Test-Sendungen generiert. Zeitexpandiertes Netzwerk erfolgreich aufgebaut.")

## 4. Durchführung des Benchmarks

Wir führen das sequentielle Greedy-Routing nacheinander mit dem standardmäßigen `DijkstraRouter` und dem optimierten `AStarRouter` aus. Wir messen jeweils die Gesamtlaufzeit und überprüfen, ob die berechneten Zielfunktionswerte mathematisch identisch sind.

In [ ]:
weights_default = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)

# 1. Dijkstra-Router benchmarken
print("Starte Dijkstra-Router (ohne Heuristik)...")
dijkstra_router = DijkstraRouter(objective_weights=weights_default)
t0 = time.time()
dijkstra_result = dijkstra_router.solve_multiple(network, show_progress=False)
dijkstra_time = time.time() - t0
print(f"Dijkstra abgeschlossen in {dijkstra_time:.2f} Sekunden. Obj-Wert: {dijkstra_result.objective_value:.6f}")

# 2. A*-Router benchmarken
print("\nStarte A*-Router (mit Heuristik)...")
astar_router = AStarRouter(objective_weights=weights_default)
t1 = time.time()
astar_result = astar_router.solve_multiple(network, show_progress=False)
astar_time = time.time() - t1
print(f"A* abgeschlossen in {astar_time:.2f} Sekunden. Obj-Wert: {astar_result.objective_value:.6f}")

# 3. Verifikation
obj_diff = abs(dijkstra_result.objective_value - astar_result.objective_value)
gap = (obj_diff / dijkstra_result.objective_value) * 100 if dijkstra_result.objective_value > 0 else 0.0
speedup = dijkstra_time / astar_time

print("\n======================================================")
print("               BENCHMARK VERIFIKATION                 ")
print("======================================================")
print(f"Dijkstra Rechenzeit:   {dijkstra_time:.2f} s")
print(f"A* Rechenzeit:         {astar_time:.2f} s")
print(f"Speedup-Faktor:        {speedup:.2f}x ({((dijkstra_time - astar_time) / dijkstra_time)*100:.1f}% schneller)")
print(f"Dijkstra Zielwert:     {dijkstra_result.objective_value:.6f}")
print(f"A* Zielwert:           {astar_result.objective_value:.6f}")
print(f"Optimality Gap:        {gap:.6f} %")
print(f"Mathematisch identisch: {obj_diff < 1e-9}")
print("======================================================")

## 5. Visualisierung des Performance-Vergleichs

In [ ]:
df_compare = pd.DataFrame({
    "Router": ["Dijkstra-Router\n(Standard)", "A*-Router\n(Optimiert)"],
    "Rechenzeit (s)": [dijkstra_time, astar_time]
})

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_compare, x="Router", y="Rechenzeit (s)", palette=["#9b59b6", "#1abc9c"])
plt.title(f"Performance-Vergleich für {N_SHIPMENTS} Sendungen (30 Tage Horizon)", fontsize=13, fontweight="bold")
plt.ylabel("Gesamtlaufzeit (s)", fontsize=11)
plt.xlabel("", fontsize=11)

# Annotiere Werte auf den Balken
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f} s", 
                (p.get_x() + p.get_width() / 2., p.get_height() / 2.), 
                ha='center', va='center', 
                xytext=(0, 0), 
                textcoords='offset points', 
                fontsize=12, fontweight="bold", color="white")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "dataset/dijkstra_vs_astar_comparison.png", dpi=300)
plt.show()